# Experiment 09 — From Scratch: LN 7×7 vs BN 7×7 vs LN 3×3 vs BN 3×3

| Config | Norm | DWConv | Pretrain |
|--------|------|--------|----------|
| 09a | LayerNorm | 7×7 | ❌ |
| 09b | BatchNorm | 7×7 | ❌ |
| 09c | LayerNorm | 3×3 | ❌ |
| 09d | BatchNorm | 3×3 | ❌ |

**Comparisons yang bisa dibaca:**
- 09a vs 09b → apple to apple: LN vs BN pada 7×7 (normalization only)
- 09c vs 09d → apple to apple: LN vs BN pada 3×3 (normalization only)
- 09a vs 09c → isolasi efek kernel size pada LN (7×7 vs 3×3)
- 09b vs 09d → isolasi efek kernel size pada BN (7×7 vs 3×3)

**Extension of Finding 1:** apakah gap LN vs BN mengecil tanpa pretrained weights?

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# ── Sesuaikan path dataset di Drive ──────────────────────────────────────────
DATASET_DRIVE = '/content/drive/MyDrive/rice-convnext/dataset'
DATASET_LOCAL = '/content/dataset'
DRIVE_DIR     = '/content/drive/MyDrive/rice-convnext/09_Scratch_LN7x7_BN3x3_LN3x3'
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(DATASET_LOCAL):
    print('Copying dataset ke local SSD...')
    shutil.copytree(DATASET_DRIVE, DATASET_LOCAL)
    print('Done.')
else:
    print('Local dataset sudah ada.')

In [ ]:
import random, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
IMG_SIZE      = 224
BATCH_SIZE    = 64
NUM_EPOCHS    = 30
LR_INIT       = 1e-3
LR_FINAL      = 1e-6
WEIGHT_DECAY  = 0.05
NUM_WORKERS   = 2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CONFIGS = {
    '09a': {'name': 'LN 7×7 | No Pretrain', 'norm': 'LN', 'kernel': 7,
            'save': f'{DRIVE_DIR}/09a_ln7x7.pt'},
    '09b': {'name': 'BN 7×7 | No Pretrain', 'norm': 'BN', 'kernel': 7,
            'save': f'{DRIVE_DIR}/09b_bn7x7.pt'},
    '09c': {'name': 'LN 3×3 | No Pretrain', 'norm': 'LN', 'kernel': 3,
            'save': f'{DRIVE_DIR}/09c_ln3x3.pt'},
    '09d': {'name': 'BN 3×3 | No Pretrain', 'norm': 'BN', 'kernel': 3,
            'save': f'{DRIVE_DIR}/09d_bn3x3.pt'},
}
print('Configs:', list(CONFIGS.keys()))

## 2. Data Loading

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

base        = datasets.ImageFolder(DATASET_LOCAL)
CLASS_NAMES = base.classes
NUM_CLASSES = len(CLASS_NAMES)
N           = len(base)
n_train, n_val = int(0.6 * N), int(0.2 * N)
n_test      = N - n_train - n_val

gen = torch.Generator().manual_seed(SEED)
tr_idx, val_idx, te_idx = [
    s.indices for s in random_split(base, [n_train, n_val, n_test], generator=gen)
]

train_ds = Subset(datasets.ImageFolder(DATASET_LOCAL, transform=train_tf), tr_idx)
val_ds   = Subset(datasets.ImageFolder(DATASET_LOCAL, transform=val_tf),   val_idx)
test_ds  = Subset(datasets.ImageFolder(DATASET_LOCAL, transform=val_tf),   te_idx)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Classes: {NUM_CLASSES} | Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 3. Architecture Helpers

In [ ]:
class BatchNormNHWC(nn.Module):
    """BatchNorm2d that accepts NHWC input (ConvNeXt block format)."""
    def __init__(self, num_features):
        super().__init__()
        self.bn = nn.BatchNorm2d(num_features)
    def forward(self, x):
        return self.bn(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)


def replace_layernorm_with_batchnorm(module):
    for name, child in module.named_children():
        if type(child).__name__ == 'LayerNorm2d':
            setattr(module, name, nn.BatchNorm2d(child.normalized_shape[0]))
        elif isinstance(child, nn.LayerNorm):
            setattr(module, name, BatchNormNHWC(child.normalized_shape[0]))
        else:
            replace_layernorm_with_batchnorm(child)


def replace_dwconv_kernel(module, new_kernel=3):
    for name, child in module.named_children():
        if (isinstance(child, nn.Conv2d)
                and child.kernel_size == (7, 7)
                and child.groups == child.in_channels):
            setattr(module, name, nn.Conv2d(
                child.in_channels, child.out_channels,
                kernel_size=new_kernel, padding=new_kernel // 2,
                groups=child.groups, bias=child.bias is not None,
            ))
        else:
            replace_dwconv_kernel(child, new_kernel)


def build_model(norm='LN', kernel=7):
    model = models.convnext_tiny(weights=None)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
    if kernel != 7:
        replace_dwconv_kernel(model, new_kernel=kernel)
    if norm == 'BN':
        replace_layernorm_with_batchnorm(model)
    return model.to(DEVICE)


print('Architecture helpers defined.')

## 4. Training & Evaluation

In [ ]:
def get_scheduler(opt):
    warmup = optim.lr_scheduler.LinearLR(opt, start_factor=0.01, end_factor=1.0, total_iters=1)
    cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS - 1, eta_min=LR_FINAL)
    return optim.lr_scheduler.SequentialLR(opt, schedulers=[warmup, cosine], milestones=[1])


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            if training:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += out.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total


def train(cfg_key, cfg):
    model     = build_model(norm=cfg['norm'], kernel=cfg['kernel'])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer)
    best_acc  = 0.0
    history   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f'\n[{cfg_key}] {cfg["name"]}')
    print('─' * 68)
    for epoch in range(1, NUM_EPOCHS + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader,   criterion)
        scheduler.step()
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(vl);   history['val_acc'].append(va)
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), cfg['save'])
        print(f'  Epoch {epoch:02d}/{NUM_EPOCHS} | '
              f'Train Loss: {tl:.4f} Acc: {ta:.4f} | '
              f'Val Loss: {vl:.4f} Acc: {va:.4f} | Best: {best_acc:.4f}')

    return model, history


def evaluate(model, cfg):
    model.load_state_dict(torch.load(cfg['save'], map_location=DEVICE))
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            preds.extend(model(imgs.to(DEVICE)).max(1)[1].cpu().numpy())
            labels.extend(lbls.numpy())
    report = classification_report(labels, preds, target_names=CLASS_NAMES, output_dict=True)
    cm     = confusion_matrix(labels, preds)
    print(f'  Accuracy: {report["accuracy"]:.4f} | F1: {report["weighted avg"]["f1-score"]:.4f}')
    return {
        'accuracy': report['accuracy'],
        'f1'      : report['weighted avg']['f1-score'],
        'report'  : report,
        'cm_norm' : cm.astype('float') / cm.sum(axis=1)[:, np.newaxis],
        'preds'   : preds, 'labels': labels,
    }


print('Training functions defined.')

## 5. Run All 4 Experiments

In [ ]:
# ── 09a: LN 7×7 ───────────────────────────────────────────────────────────────
model_a, history_a = train('09a', CONFIGS['09a'])
print('\nEvaluating 09a...')
results_a = evaluate(model_a, CONFIGS['09a'])

In [ ]:
# ── 09b: BN 7×7 ───────────────────────────────────────────────────────────────
model_b, history_b = train('09b', CONFIGS['09b'])
print('\nEvaluating 09b...')
results_b = evaluate(model_b, CONFIGS['09b'])

In [ ]:
# ── 09c: LN 3×3 ───────────────────────────────────────────────────────────────
model_c, history_c = train('09c', CONFIGS['09c'])
print('\nEvaluating 09c...')
results_c = evaluate(model_c, CONFIGS['09c'])

In [ ]:
# ── 09d: BN 3×3 ───────────────────────────────────────────────────────────────
model_d, history_d = train('09d', CONFIGS['09d'])
print('\nEvaluating 09d...')
results_d = evaluate(model_d, CONFIGS['09d'])

In [ ]:
# ── Summary table + gap analysis ──────────────────────────────────────────────
EXP01_ACC, EXP02_ACC = 0.9137, 0.7679   # pretrained LN vs BN gap reference

rows = [
    ('Exp 01 (LN 7×7, Pretrained)', EXP01_ACC, None),
    ('Exp 02 (BN 7×7, Pretrained)', EXP02_ACC, None),
    ('09a    (LN 7×7, Scratch)',     results_a['accuracy'], results_a['f1']),
    ('09b    (BN 7×7, Scratch)',     results_b['accuracy'], results_b['f1']),
    ('09c    (LN 3×3, Scratch)',     results_c['accuracy'], results_c['f1']),
    ('09d    (BN 3×3, Scratch)',     results_d['accuracy'], results_d['f1']),
]

print(f'{"Config":<35} {"Accuracy":>10} {"F1":>10}')
print('─' * 57)
for name, acc, f1 in rows:
    f1_str = f'{f1:.4f}' if f1 else '—'
    print(f'{name:<35} {acc:.4f}     {f1_str:>10}')

print('─' * 57)
print(f'\nGap analysis:')
print(f'  Pretrained  LN vs BN (7×7):  {EXP01_ACC - EXP02_ACC:+.4f}  (exp 01 vs 02) ← reference')
print(f'  Scratch     LN vs BN (7×7):  {results_a["accuracy"] - results_b["accuracy"]:+.4f}  (09a vs 09b) ← apple to apple')
print(f'  Scratch     LN vs BN (3×3):  {results_c["accuracy"] - results_d["accuracy"]:+.4f}  (09c vs 09d) ← apple to apple')
print(f'  Scratch     LN 7×7 vs 3×3:  {results_a["accuracy"] - results_c["accuracy"]:+.4f}  (09a vs 09c) ← kernel effect on LN')
print(f'  Scratch     BN 7×7 vs 3×3:  {results_b["accuracy"] - results_d["accuracy"]:+.4f}  (09b vs 09d) ← kernel effect on BN')

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
labels = ['Exp 01\nLN 7×7\nPretrained', 'Exp 02\nBN 7×7\nPretrained',
          '09a\nLN 7×7\nScratch', '09b\nBN 7×7\nScratch',
          '09c\nLN 3×3\nScratch', '09d\nBN 3×3\nScratch']
accs   = [EXP01_ACC, EXP02_ACC,
          results_a['accuracy'], results_b['accuracy'],
          results_c['accuracy'], results_d['accuracy']]
colors = ['royalblue', 'tomato', 'steelblue', 'salmon', 'mediumseagreen', 'darksalmon']

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(labels, accs, color=colors, edgecolor='black', width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.axvline(x=1.5, color='gray', ls='--', alpha=0.6)
ax.text(0.5, 1.06, 'Pretrained', ha='center', fontsize=11, color='royalblue', fontweight='bold')
ax.text(3.5, 1.06, 'From Scratch', ha='center', fontsize=11, color='steelblue', fontweight='bold')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Exp 09: LN vs BN — Pretrained vs Scratch | Kernel Size Ablation', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_bar_09.png', dpi=150)
plt.show()

In [ ]:
# ── Overlaid val accuracy curves ──────────────────────────────────────────────
ep = range(1, NUM_EPOCHS + 1)
plt.figure(figsize=(10, 5))
plt.plot(ep, history_a['val_acc'], label='09a LN 7×7', color='steelblue',     lw=2)
plt.plot(ep, history_b['val_acc'], label='09b BN 7×7', color='salmon',        lw=2)
plt.plot(ep, history_c['val_acc'], label='09c LN 3×3', color='mediumseagreen',lw=2)
plt.plot(ep, history_d['val_acc'], label='09d BN 3×3', color='darksalmon',    lw=2)
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy')
plt.title('Exp 09 — Val Accuracy (From Scratch)', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_curves_09.png', dpi=150)
plt.show()

In [ ]:
# ── 4-way confusion matrices ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(24, 18))
for ax, res, cfg in zip(axes.flatten(),
                        [results_a, results_b, results_c, results_d],
                        ['09a LN 7×7', '09b BN 7×7', '09c LN 3×3', '09d BN 3×3']):
    sns.heatmap(res['cm_norm'], annot=True, fmt='.2f',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cmap='Blues', linewidths=0.5, ax=ax)
    ax.set_title(f'{cfg}\nAcc={res["accuracy"]:.4f}', fontsize=11, fontweight='bold')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('Exp 09 — Confusion Matrices (From Scratch)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_cm_09.png', dpi=150)
plt.show()

## 7. Save to Drive

In [ ]:
for fname in ['comparison_bar_09.png', 'comparison_curves_09.png', 'comparison_cm_09.png']:
    if os.path.exists(fname):
        shutil.copy(fname, f'{DRIVE_DIR}/{fname}')
        print(f'✅  {fname}')

# Save per-config classification reports
import pandas as pd
for key, res in [('09a', results_a), ('09b', results_b), ('09c', results_c), ('09d', results_d)]:
    df = pd.DataFrame(res['report']).transpose()
    fname = f'classification_report_{key}.csv'
    df.to_csv(fname)
    shutil.copy(fname, f'{DRIVE_DIR}/{fname}')
    print(f'✅  {fname}')

print('\nDone.')